In [44]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import json

In [45]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [46]:
with open('../data/processing/not_fully/laptops.csv', 'r', encoding='utf-8') as f:
    df1 = pd.read_csv(f)

with open('../data/processing/not_fully/smartphones.csv', 'r', encoding='utf-8') as f:
    df2 = pd.read_csv(f)

with open('../data/processing/not_fully/tablets.csv', 'r', encoding='utf-8') as f:
    df3 = pd.read_csv(f)

In [47]:
df1.dtypes

category                     object
name_card                    object
price                       float64
price_old                   float64
price_installment           float64
rating_card                 float64
reviews_card                float64
name_seller                  object
rating_seller               float64
URL                          object
cpu                          object
ssd_capacity                float64
gpu_full_name                object
os                           object
screen_diagonal             float64
ram                         float64
auto_time                   float64
weight_kg                   float64
matrix                       object
ram_type                     object
case_material                object
cpu_model                    object
processor_brand              object
ghz_processor               float64
processor_cores             float64
screen_size                  object
brightness                  float64
touch_screen                

In [48]:
df1.isnull().sum()

category                       0
name_card                    979
price                       2873
price_old                   2873
price_installment            998
rating_card                 3000
reviews_card                2976
name_seller                  979
rating_seller               1181
URL                            0
cpu                          852
ssd_capacity                 624
gpu_full_name               1058
os                           680
screen_diagonal              587
ram                          464
auto_time                   2861
weight_kg                    948
matrix                      1076
ram_type                     986
case_material               1592
cpu_model                    455
processor_brand              631
ghz_processor                977
processor_cores              776
screen_size                  688
brightness                  3408
touch_screen                1993
battery_capacity            2065
screen_coating              1136
gpu_type  

In [79]:
df1.sample(5)

,category,name_card,price,price_old,price_installment,rating_card,reviews_card,name_seller,rating_seller,cpu,ssd_capacity,gpu_full_name,os,screen_diagonal,ram,auto_time,weight_kg,matrix,ram_type,case_material,cpu_model,processor_brand,ghz_processor,processor_cores,screen_size,touch_screen,battery_capacity,screen_coating,gpu_type,brand_graphic_proccessor,storage_type,hz_screen,web_camera,discount_abs,discount_pct,ram_per_core,high_refresh_rate,cpu_power_proxy,res_w,res_h,pixel_count
1990,laptop,MSI Cyborg 15 B2RWFKG-025XRU Игровой ноутбук 1...,652385.0,863095.0,54366.0,5.0,8.0,Ozon Россия,4.9,Intel Core 5 210H,1024.0,NVIDIA GeForce RTX 5060 (8 Гб),Без системы,15.6,16.0,7.0,1.70,IPS,no_data,no_data,Intel Core 5,Intel,2.2,8.0,1920x1080,Нет,56.0,no_data,Дискретная,NVIDIA,SSD,144.0,2.0,210710.0,24.41,2.000000,True,17.6,1920.0,1080.0,2073600.0
543,laptop,"RZUC 6006U Ноутбук 15.6"", Intel Core i3-6006U,...",178582.0,322013.0,14882.0,4.7,219.0,RZUC цифровая техни...,4.7,Intel Core i3-6006U,2048.0,Intel UHD Graphics,Windows Pro,15.6,32.0,7.0,2.10,IPS,DDR4,"SBS пластик, ABS пластик",Intel Core i3,Intel,2.0,4.0,1920x1080,Нет,96.0,Антибликовое,Встроенная,Intel,SSD,60.0,3.0,143431.0,44.54,8.000000,False,8.0,1920.0,1080.0,2073600.0
3860,laptop,NaN,424220.0,834731.0,37342.0,4.9,30.0,no_data,4.8,no_data,512.0,AMD Radeon,no_data,15.6,16.0,7.0,1.70,no_data,no_data,no_data,no_data,no_data,5.0,6.0,no_data,no_data,56.0,no_data,no_data,no_data,no_data,165.0,2.0,410511.0,49.18,2.666667,True,30.0,1920.0,1080.0,2073600.0
2175,laptop,"Redmi Book 16 JYU4640CN Ноутбук 16"", Intel Cor...",1003291.0,3582750.0,83608.0,5.0,150.0,ElectroShop,4.9,Intel Core 5 220H,512.0,Intel Iris Xe Graphics,Windows Home,16.0,16.0,7.0,1.65,IPS,LPDDR5X,Металл,Intel Core 5,Intel,3.7,12.0,2560x1600,no_data,72.0,Матовое,Встроенная,Intel,SSD,120.0,2.0,2579459.0,72.00,1.333333,True,44.4,2560.0,1600.0,4096000.0
1113,laptop,NaN,424220.0,834731.0,37342.0,4.9,30.0,no_data,4.8,no_data,1000.0,no_data,no_data,15.6,24.0,7.0,1.70,no_data,no_data,no_data,Apple M4 Pro,no_data,2.6,8.0,no_data,no_data,56.0,no_data,no_data,no_data,no_data,60.0,2.0,410511.0,49.18,3.000000,False,20.8,1920.0,1080.0,2073600.0


In [50]:
# Диагностика столбцов для возможного удаления
n_rows = len(df1)
summary = pd.DataFrame({
    'null_count': df1.isna().sum(),
    'null_pct': (df1.isna().mean() * 100).round(2),
    'nunique': df1.nunique(dropna=False),
    'dtype': df1.dtypes.astype(str)
}).sort_values('null_pct', ascending=False)

print('Rows:', n_rows)
print('Columns:', df1.shape[1])
print('\nTop columns by null %:')
print(summary.head(20))

const_cols = summary[summary['nunique'] <= 1].index.tolist()
high_null_cols = summary[summary['null_pct'] >= 70].index.tolist()

print('\nConstant/near-constant cols (nunique <= 1):', const_cols)
print('High-null cols (>=70% NaN):', high_null_cols)

Rows: 5376
Columns: 39

Top columns by null %:
                  null_count  null_pct  nunique    dtype
hdd_capacity            5143     95.67       18  float64
video_memory            4098     76.23       13  float64
chipset                 3763     70.00       19   object
brightness              3408     63.39       31  float64
brand                   3325     61.85       60   object
rating_card             3000     55.80       32  float64
reviews_card            2976     55.36      258  float64
price                   2873     53.44     2408  float64
price_old               2873     53.44     2341  float64
auto_time               2861     53.22       68  float64
battery_capacity        2065     38.41      144  float64
touch_screen            1993     37.07        3   object
web_camera              1955     36.37       12  float64
case_material           1592     29.61       85   object
hz_screen               1323     24.61       14  float64
rating_seller           1181     21.97   

In [51]:
df1.drop(columns=['hdd_capacity','URL','video_memory','chipset','brightness','brand'], inplace=True)

In [52]:
df1.isna().sum()

category                       0
name_card                    979
price                       2873
price_old                   2873
price_installment            998
rating_card                 3000
reviews_card                2976
name_seller                  979
rating_seller               1181
cpu                          852
ssd_capacity                 624
gpu_full_name               1058
os                           680
screen_diagonal              587
ram                          464
auto_time                   2861
weight_kg                    948
matrix                      1076
ram_type                     986
case_material               1592
cpu_model                    455
processor_brand              631
ghz_processor                977
processor_cores              776
screen_size                  688
touch_screen                1993
battery_capacity            2065
screen_coating              1136
gpu_type                    1131
brand_graphic_proccessor    1030
storage_ty

In [53]:
median_col = ['price','price_old','price_installment','rating_card','reviews_card','ssd_capacity','screen_diagonal','ram',
           'auto_time','weight_kg','ghz_processor','processor_cores','battery_capacity','hz_screen']
for col in median_col:
    df1[col] = df1[col].fillna(df1[col].median())

mode_col = ['rating_seller','web_camera']
for col in mode_col:
    df1[col] = df1[col].fillna(df1[col].mode()[0])

In [54]:
cat_cols = ['name_seller','cpu','gpu_full_name','os','matrix','ram_type','case_material','cpu_model',''
            'processor_brand','screen_size','touch_screen','screen_coating', 'gpu_type', 'brand_graphic_proccessor','storage_type']
for col in cat_cols:
    df1[col] = df1[col].fillna('no_data')

In [55]:
cat_cols

['name_seller',
 'cpu',
 'gpu_full_name',
 'os',
 'matrix',
 'ram_type',
 'case_material',
 'cpu_model',
 'processor_brand',
 'screen_size',
 'touch_screen',
 'screen_coating',
 'gpu_type',
 'brand_graphic_proccessor',
 'storage_type']

In [56]:
df1.isna().sum()

category                      0
name_card                   979
price                         0
price_old                     0
price_installment             0
rating_card                   0
reviews_card                  0
name_seller                   0
rating_seller                 0
cpu                           0
ssd_capacity                  0
gpu_full_name                 0
os                            0
screen_diagonal               0
ram                           0
auto_time                     0
weight_kg                     0
matrix                        0
ram_type                      0
case_material                 0
cpu_model                     0
processor_brand               0
ghz_processor                 0
processor_cores               0
screen_size                   0
touch_screen                  0
battery_capacity              0
screen_coating                0
gpu_type                      0
brand_graphic_proccessor      0
storage_type                  0
hz_scree

In [80]:
df2.sample(5)

,category,name_card,price,price_old,price_installment,rating_card,reviews_card,name_seller,rating_seller,article,screen_diagonal,battery_capacity,cpu,screen_size,processor_cores,ram,built-in_memory,main_camera,os,hz_screen,matrix,processor_brand,ghz_processor,e_sim,front_camera,type_of_frame,degree_of_protect,dimensions,weight_g,country_of_origin,case_material,brand_graphic_proccessor,video_processor,max_fps_video,discount_abs,discount_pct,cpu_power_proxy,memory_per_price,ram_per_price,total_camera_mp,selfie_ratio,battery_per_gram,high_fps_video,esim_flag,protection_flag,res_w,res_h,pixel_count
3382,smartphones,"Смартфон 15C Global 8/256 ГБ, черный",81823.0,172689.0,6819.0,4.9,27.0,OOO SMARTSTORE,4.2,3.184204e+09,6.90,6000.0,Helio G81 Ultra,1600x720,8.0,8.0,256.0,50.0,Android,120.0,IPS,MediaTek,2.0,SIM (без eSIM),8.0,Моноблок,IP64,171.5х79.5х8,205.0,Китай,Пластик,ARM,Mali-G52 MC2,30.0,90866.0,52.618291,16.0,0.003129,0.000098,58.0,0.160000,29.268293,0,0,1,1600.0,720.0,1152000.0
2144,smartphones,"Xiaomi Смартфон poco x5 pro Global 12/256 ГБ, ...",90081.0,401659.0,7507.0,4.8,42.0,Mi Future Hub,4.7,2.951605e+09,6.67,5000.0,Snapdragon 778G,2400x1080,8.0,12.0,256.0,108.0,Android,120.0,AMOLED,Qualcomm,2.4,SIM (без eSIM),16.0,Моноблок,IP68,160*75*8,200.0,"Китай, Китай (Гонконг)",Стекло,Qualcomm,Adreno 642L,60.0,311578.0,77.572767,19.2,0.002842,0.000133,124.0,0.148148,25.000000,1,0,1,2400.0,1080.0,2592000.0
1670,smartphones,"Смартфон Хонор i17 Смартфон, противоударный на...",47303.0,320478.0,3942.0,4.6,123.0,chanson,4.6,3.557950e+09,6.80,6000.0,no_data,1920x1080,8.0,12.0,512.0,200.0,Android,120.0,IPS,AMD,2.0,SIM (без eSIM),50.0,Моноблок,IP66,160*90*60,400.0,Китай,Металл,Adreno,no_data,60.0,273175.0,85.239860,16.0,0.010824,0.000254,250.0,0.250000,15.000000,1,0,1,1920.0,1080.0,2073600.0
4915,smartphones,"Redmi Смартфон 8/256 ГБ, белый, голубой",53388.0,198288.0,4449.0,4.4,748.0,Телефон для Семьи,4.6,3.461561e+09,6.88,6000.0,no_data,1612x720,2.0,8.0,256.0,50.0,Android,90.0,IPS,MediaTek,2.0,no_data,13.0,Моноблок,IP67,"171,9 x 78 x 8 ,2",207.0,Китай,Пластик,no_data,no_data,60.0,144900.0,73.075527,4.0,0.004795,0.000150,63.0,0.260000,28.985507,1,0,1,1612.0,720.0,1160640.0
2568,smartphones,"W&O Смартфон Ростест (EAC) 4/64 ГБ, светло-серый",21881.0,184733.0,1824.0,4.2,40.0,Мир электроники Маг...,4.4,2.952073e+09,4.50,2800.0,MT6735M,1440x720,4.0,4.0,64.0,5.0,Android,60.0,no_data,MediaTek,2.4,SIM (без eSIM),5.0,Моноблок,IP58,110*52*11,128.0,Китай,"Пластик, Металл",no_data,no_data,60.0,162852.0,88.155338,9.6,0.002925,0.000183,10.0,1.000000,21.875000,1,0,1,1440.0,720.0,1036800.0


In [58]:
df2.rename(columns={'Characteristics.Встроенная память':'built-in_memory'}, inplace=True)

In [59]:

n_rows = len(df2)
summary = pd.DataFrame({
    'null_count': df2.isna().sum(),
    'null_pct': (df2.isna().mean() * 100).round(2),
    'nunique': df2.nunique(dropna=False),
    'dtype': df2.dtypes.astype(str)
}).sort_values('null_pct', ascending=False)

print('Rows:', n_rows)
print('Columns:', df1.shape[1])
print('\nTop columns by null %:')
print(summary.head(20))

const_cols = summary[summary['nunique'] <= 1].index.tolist()
high_null_cols = summary[summary['null_pct'] >= 70].index.tolist()

print('\nConstant/near-constant cols (nunique <= 1):', const_cols)
print('High-null cols (>=70% NaN):', high_null_cols)

Rows: 5382
Columns: 33

Top columns by null %:
                          null_count  null_pct  nunique    dtype
harmonyos_version               5318     98.81        7   object
ios_version                     5024     93.35       13   object
brand                           3811     70.81       28   object
video_processor                 1941     36.06      122   object
max_fps_video                   1935     35.95       14  float64
brand_graphic_proccessor        1607     29.86       16   object
hz_screen                       1370     25.46       10  float64
price                           1350     25.08     3171  float64
price_old                       1350     25.08     2319  float64
degree_of_protect               1003     18.64       73   object
android_version                  954     17.73       17   object
cpu                              772     14.34      228   object
e_sim                            713     13.25        4   object
ghz_processor                    581     10

In [60]:
df2.isna().sum()

category                       0
name_card                    312
price                       1350
price_old                   1350
price_installment            325
rating_card                  530
reviews_card                 530
name_seller                  312
rating_seller                315
url                            0
article                        9
screen_diagonal              117
battery_capacity             231
cpu                          772
screen_size                  285
processor_cores              209
ram                          161
built-in_memory               76
main_camera                  223
os                           131
hz_screen                   1370
matrix                       476
processor_brand              337
ghz_processor                581
e_sim                        713
front_camera                 244
type_of_frame                509
degree_of_protect           1003
dimensions                   274
weight_g                     302
country_of

In [61]:
cat_cols = df2.select_dtypes(exclude=[np.number]).columns.tolist()
cat_cols

['category',
 'name_card',
 'name_seller',
 'url',
 'cpu',
 'screen_size',
 'os',
 'matrix',
 'processor_brand',
 'e_sim',
 'type_of_frame',
 'degree_of_protect',
 'dimensions',
 'country_of_origin',
 'case_material',
 'android_version',
 'brand_graphic_proccessor',
 'video_processor',
 'brand',
 'ios_version',
 'harmonyos_version']

In [62]:
df2.drop(columns=['harmonyos_version','ios_version','brand','url','android_version'], inplace=True)

In [63]:
med_col = ['price','price_old','rating_card','reviews_card','price_installment','rating_seller','screen_diagonal','ram',
           'weight_g','built-in_memory','ghz_processor','processor_cores','battery_capacity','hz_screen','main_camera','front_camera','max_fps_video']

for col in med_col:
    df2[col] = df2[col].fillna(df2[col].median())

unk_col = ['category', 'name_seller','cpu', 'screen_size','os','matrix','processor_brand','e_sim','type_of_frame',
           'degree_of_protect','dimensions','country_of_origin','case_material','brand_graphic_proccessor','video_processor']
for col in unk_col:
    df2[col] = df2[col].fillna('no_data')

In [64]:
df2.isna().sum()

category                      0
name_card                   312
price                         0
price_old                     0
price_installment             0
rating_card                   0
reviews_card                  0
name_seller                   0
rating_seller                 0
article                       9
screen_diagonal               0
battery_capacity              0
cpu                           0
screen_size                   0
processor_cores               0
ram                           0
built-in_memory               0
main_camera                   0
os                            0
hz_screen                     0
matrix                        0
processor_brand               0
ghz_processor                 0
e_sim                         0
front_camera                  0
type_of_frame                 0
degree_of_protect             0
dimensions                    0
weight_g                      0
country_of_origin             0
case_material                 0
brand_gr

In [81]:
df3.sample(5)

,category,name_card,price,price_old,price_installment,rating_card,reviews_card,name_seller,rating_seller,url,article,screen_diagonal,ram,os,internal_storage,screen_size,cpu,main_camera,matrix,type_of_tablet,announcement_year,hz_screen,ppi,processor_brand,processor_cores,ghz_processor,brand_graphic_proccessor,video_processor,front_camera,keyboard,case_material,waterproof,battery_capacity,accessories_included,dimensions,weight_g,country_of_origin,res_w,res_h,pixel_count,discount_abs,discount_pct,cpu_power_proxy,ram_per_core,storage_per_price,ram_per_price,battery_per_gram,total_camera_mp,high_refresh_rate,screen_area_proxy,keyboard_flag,stylus_flag,waterproof_flag
3567,tablets,"Apple Планшет Смартфон Apple iPhone 17 512GB, ...",60747.5,302265.0,71475.0,4.9,33.0,iPolly,4.7,https://www.ozon.kz/product/apple-planshet-sma...,3.132292e+09,6.3,12.00,no_data,512.0,no_data,no_data,48.0,Super Retina XDR,Планшет,2025.0,120.0,264.0,no_data,6.0,2.5,no_data,no_data,12.0,no_data,"Металл, Стекло",no_data,9705.0,no_data,no_data,177.0,no_data,2560.0,1600.0,4096000.0,241517.5,79.902569,15.0,2.00000,0.008428,0.000198,54.830508,60.0,1,39.69,0,0,0
4179,tablets,Детский планшет Планшет детский Андроид для му...,60747.5,302265.0,4311.0,4.9,33.0,ИП Морозова С.О.,4.5,https://www.ozon.kz/product/detskiy-planshet-p...,3.649229e+09,7.0,0.25,Android,256.0,1920x1080,no_data,2.0,IPS,Детский планшет,2025.0,120.0,264.0,no_data,8.0,2.0,Qualcomm,Adreno 510,2.0,no_data,Металл,Да,5000.0,no_data,320*180*120,800.0,Китай,1920.0,1080.0,2073600.0,241517.5,79.902569,16.0,0.03125,0.004214,0.000004,6.250000,4.0,1,49.00,0,0,1
3752,tablets,"Redmi Планшет Redmi Pad 2 Wi-Fi+LTE, 11"" 6 ГБ/...",125748.0,190506.0,10479.0,4.9,33.0,"""ТвойМаркет Электро...",4.8,https://www.ozon.kz/product/redmi-planshet-red...,2.633959e+09,11.0,6.00,Android,128.0,2560x1600,Helio G100 Ultra,8.0,IPS,Планшет,2025.0,90.0,264.0,MediaTek,8.0,2.2,ARM,Mali-G57 MC2,5.0,Опционально,Металл,Да,9000.0,no_data,"254,4 x 166 x 7,36",510.0,Китай,2560.0,1600.0,4096000.0,64758.0,33.992630,17.6,0.75000,0.001018,0.000048,17.647059,13.0,1,121.00,1,0,1
465,tablets,"Планшет Appei Pab 17 Ultra，5G(LTE), Полный Ком...",58818.0,352850.0,4902.0,5.0,71.0,Technology Store,4.8,https://www.ozon.kz/product/planshet-appei-pab...,3.753081e+09,10.1,12.00,Android,1023.0,2560x1600,Snapdragon 8 Gen3 Leading Version,32.0,IPS,Планшет,2026.0,140.0,264.0,Qualcomm,8.0,3.4,Qualcomm,Adreno 750,16.0,Опционально,"Металл, Стекло, Пластик",Да,10000.0,Клавиатура,214*160*8.7,300.0,Китай,2560.0,1600.0,4096000.0,294032.0,83.330594,27.2,1.50000,0.017393,0.000204,33.333333,48.0,1,102.01,1,0,1
711,tablets,"HUAWEI Планшет MatePad SE 11 Wi-Fi, 11.0"" 4 ГБ...",60747.5,302265.0,7711.0,4.7,15.0,Абсолют Трейд,4.9,https://www.ozon.kz/product/huawei-planshet-ma...,2.396642e+09,11.0,4.00,HarmonyOS,128.0,1920x1200,no_data,8.0,IPS,Планшет,2024.0,60.0,207.0,HiSilicon,8.0,2.5,ARM,Mali-G51 MP4,5.0,no_data,Металл,Нет,7700.0,no_data,252.3*163.8*6.9,475.0,Китай,1920.0,1200.0,2304000.0,241517.5,79.902569,20.0,0.50000,0.002107,0.000066,16.210526,13.0,0,121.00,0,0,0


In [66]:
# Диагностика столбцов для возможного удаления
n_rows = len(df3)
summary = pd.DataFrame({
    'null_count': df3.isna().sum(),
    'null_pct': (df3.isna().mean() * 100).round(2),
    'nunique': df3.nunique(dropna=False),
    'dtype': df3.dtypes.astype(str)
}).sort_values('null_pct', ascending=False)

print('Rows:', n_rows)
print('Columns:', df1.shape[1])
print('\nTop columns by null %:')
print(summary.head(20))

const_cols = summary[summary['nunique'] <= 1].index.tolist()
high_null_cols = summary[summary['null_pct'] >= 70].index.tolist()

print('\nConstant/near-constant cols (nunique <= 1):', const_cols)
print('High-null cols (>=70% NaN):', high_null_cols)

Rows: 5553
Columns: 33

Top columns by null %:
                          null_count  null_pct  nunique    dtype
harmonyos_version               5317     95.75        7   object
ios_version                     5174     93.17       14   object
brand                           4504     81.11       53   object
video_processor                 2826     50.89      113   object
accessories_included            2745     49.43        5   object
ppi                             2561     46.12      115  float64
keyboard                        2126     38.29        3   object
rating_card                     2116     38.11       33  float64
reviews_card                    2084     37.53      327  float64
hz_screen                       1722     31.01       12  float64
price_old                       1691     30.45     2583  float64
price                           1691     30.45     3116  float64
cpu                             1550     27.91      178   object
brand_graphic_proccessor        1508     27

In [67]:
df3.drop(columns=['harmonyos_version','ios_version','brand','android_version'], inplace=True)

In [68]:
df3.isna().sum()

category                       0
name_card                    311
price                       1691
price_old                   1691
price_installment            332
rating_card                 2116
reviews_card                2084
name_seller                  311
rating_seller                368
url                            0
article                        3
screen_diagonal              313
ram                          824
os                           280
internal_storage             106
screen_size                  437
cpu                         1550
main_camera                  580
matrix                       906
type_of_tablet                 3
announcement_year            821
hz_screen                   1722
ppi                         2561
processor_brand              638
processor_cores              386
ghz_processor               1052
brand_graphic_proccessor    1508
video_processor             2826
front_camera                 688
keyboard                    2126
case_mater

In [69]:
num_cols_df3 = df3.select_dtypes(include=[np.number]).columns.tolist()
num_cols_df3

['price',
 'price_old',
 'price_installment',
 'rating_card',
 'reviews_card',
 'rating_seller',
 'article',
 'screen_diagonal',
 'ram',
 'internal_storage',
 'main_camera',
 'announcement_year',
 'hz_screen',
 'ppi',
 'processor_cores',
 'ghz_processor',
 'front_camera',
 'battery_capacity',
 'weight_g']

In [70]:
cat_cols_df3 = df3.select_dtypes(exclude=[np.number]).columns.tolist()
cat_cols_df3

['category',
 'name_card',
 'name_seller',
 'url',
 'os',
 'screen_size',
 'cpu',
 'matrix',
 'type_of_tablet',
 'processor_brand',
 'brand_graphic_proccessor',
 'video_processor',
 'keyboard',
 'case_material',
 'waterproof',
 'accessories_included',
 'dimensions',
 'country_of_origin']

In [71]:
num_cols_df3 = ['price','price_old','price_installment','rating_card','reviews_card','rating_seller','screen_diagonal','ram','internal_storage',
                'main_camera','announcement_year','hz_screen','ppi','processor_cores','ghz_processor','front_camera','battery_capacity','weight_g']

for col in num_cols_df3:
    df3[col] = df3[col].fillna(df3[col].median())

cat_cols_df3 = ['category','name_seller','os','screen_size','cpu','matrix','type_of_tablet','processor_brand','brand_graphic_proccessor',
                'video_processor','keyboard','case_material','waterproof','accessories_included','dimensions','country_of_origin']

for col in cat_cols_df3:
    df3[col] = df3[col].fillna('no_data')

In [72]:
df3.isna().sum()

category                      0
name_card                   311
price                         0
price_old                     0
price_installment             0
rating_card                   0
reviews_card                  0
name_seller                   0
rating_seller                 0
url                           0
article                       3
screen_diagonal               0
ram                           0
os                            0
internal_storage              0
screen_size                   0
cpu                           0
main_camera                   0
matrix                        0
type_of_tablet                0
announcement_year             0
hz_screen                     0
ppi                           0
processor_brand               0
processor_cores               0
ghz_processor                 0
brand_graphic_proccessor      0
video_processor               0
front_camera                  0
keyboard                      0
case_material                 0
waterpro

In [73]:
df1.head()

,category,name_card,price,price_old,price_installment,rating_card,reviews_card,name_seller,rating_seller,cpu,ssd_capacity,gpu_full_name,os,screen_diagonal,ram,auto_time,weight_kg,matrix,ram_type,case_material,cpu_model,processor_brand,ghz_processor,processor_cores,screen_size,touch_screen,battery_capacity,screen_coating,gpu_type,brand_graphic_proccessor,storage_type,hz_screen,web_camera
0,laptop,"MAIBENBEN Ноутбук 15.6"", Intel Core i5-12450H,...",424220.0,834731.0,25110.0,5.0,2.0,Sotomania,4.8,Intel Core i5-12450H,512.0,Intel UHD Graphics,Windows Pro,15.6,8.0,8.5,1.75,IPS,DDR4,Алюминий,Intel Core i5,Intel,2.0,8.0,1920x1080,Нет,69.3,no_data,no_data,no_data,no_data,60.0,2.0
1,laptop,"HUAWEI MateBook 14 FLMH-X Ноутбук 14.2"", Intel...",523339.0,2251014.0,43612.0,5.0,14.0,Фирменный магазин H...,4.9,Intel Core Ultra 5 125H,1024.0,Intel Arc Graphics,Без системы,14.2,16.0,19.0,1.31,OLED,LPDDR5,Алюминий,Intel Core Ultra 5,Intel,2.6,14.0,2880x1920,Да,70.0,Глянцевое,Встроенная,Intel,SSD,120.0,2.0
2,laptop,"Apple Ноутбук 11.6"", RAM 4 ГБ, SSD 128 ГБ, Mac...",424220.0,834731.0,27878.0,4.9,30.0,Comp+,4.9,no_data,128.0,no_data,MacOS,11.6,4.0,7.0,1.70,no_data,no_data,no_data,Intel Core i5,no_data,2.6,8.0,no_data,no_data,56.0,no_data,no_data,no_data,SSD,60.0,2.0
3,laptop,Ноутбук,424220.0,834731.0,19765.0,4.3,3.0,Полюс,4.9,no_data,512.0,no_data,no_data,15.6,16.0,7.0,1.70,no_data,no_data,no_data,no_data,no_data,2.6,8.0,no_data,no_data,56.0,no_data,no_data,no_data,no_data,60.0,2.0
4,laptop,"AMDR5 Ноутбук 15.6"", RAM 32 ГБ 2048 ГБ, AMD Ra...",257297.0,604910.0,21442.0,5.0,13.0,hassenda,4.8,no_data,2048.0,AMD Radeon,Windows Pro,15.6,32.0,7.0,1.70,no_data,DDR4,no_data,AMD Ryzen 5,AMD,2.0,4.0,1920x1080,no_data,80.0,no_data,no_data,AMD,no_data,120.0,5.0


In [74]:
df1['discount_abs'] = df1['price_old'] - df1['price']
df1['discount_pct'] = ((df1['price_old'] - df1['price']) / df1['price_old'] * 100).round(2)
df1['ram_per_core'] = df1['ram'] / df1['processor_cores']
df1['high_refresh_rate'] = df1['hz_screen'] >= 120
df1['cpu_power_proxy'] = df1['ghz_processor'] * df1['processor_cores']

In [75]:
def safe_div(numerator, denominator):
    return numerator.div(denominator.replace(0, np.nan))

# Price/discount features
df2['discount_abs'] = df2['price_old'] - df2['price']
df2['discount_pct'] = safe_div(df2['discount_abs'], df2['price_old']) * 100

# Performance and value features
df2['cpu_power_proxy'] = df2['ghz_processor'] * df2['processor_cores']
df2['memory_per_price'] = safe_div(df2['built-in_memory'], df2['price'])
df2['ram_per_price'] = safe_div(df2['ram'], df2['price'])

# Camera features
df2['total_camera_mp'] = df2['main_camera'] + df2['front_camera']
df2['selfie_ratio'] = safe_div(df2['front_camera'], df2['main_camera'])

# Mobility feature
df2['battery_per_gram'] = safe_div(df2['battery_capacity'], df2['weight_g'])

# Binary flags
df2['high_fps_video'] = (df2['max_fps_video'] >= 60).astype(int)

esim_text = df2['e_sim'].astype(str).str.lower().str.strip()
df2['esim_flag'] = esim_text.str.contains('да|yes|true|есть|supported', regex=True).astype(int)

protect_text = df2['degree_of_protect'].astype(str).str.lower().str.strip()
df2['protection_flag'] = protect_text.str.contains('ip|защит|water|dust|влаг', regex=True).astype(int)

new_cols_df2 = [
    'discount_abs', 'discount_pct', 'cpu_power_proxy', 'memory_per_price',
    'ram_per_price', 'total_camera_mp', 'selfie_ratio', 'battery_per_gram',
    'high_fps_video', 'esim_flag', 'protection_flag'
]

df2[new_cols_df2].head()

,discount_abs,discount_pct,cpu_power_proxy,memory_per_price,ram_per_price,total_camera_mp,selfie_ratio,battery_per_gram,high_fps_video,esim_flag,protection_flag
0,326061.0,60.101527,20.0,0.001183,0.000037,220.0,0.100000,24.911032,1,0,1
1,386649.0,87.742162,34.0,0.018957,0.000296,158.0,0.462963,40.000000,1,0,1
2,260263.0,80.820744,16.0,0.004145,0.000130,121.0,0.120370,24.536585,0,0,1
3,260263.0,80.820744,26.4,0.008290,0.000194,82.0,0.640000,24.895833,1,0,1
4,260263.0,80.820744,16.0,0.004145,0.000130,63.0,0.260000,24.454976,0,0,1


In [76]:
def parse_screen_size(df):
    parsed = df['screen_size'].astype(str).str.extract(r'(\d+)[xхX×](\d+)', expand=True)
    df['res_w'] = pd.to_numeric(parsed[0], errors='coerce')
    df['res_h'] = pd.to_numeric(parsed[1], errors='coerce')
    df['pixel_count'] = df['res_w'] * df['res_h']
    for col in ['res_w', 'res_h', 'pixel_count']:
        df[col] = df[col].fillna(df[col].median())

for df in [df1, df2, df3]:
    parse_screen_size(df)

print('df1:', df1[['screen_size', 'res_w', 'res_h', 'pixel_count']].head(3))
print('df2:', df2[['screen_size', 'res_w', 'res_h', 'pixel_count']].head(3))
print('df3:', df3[['screen_size', 'res_w', 'res_h', 'pixel_count']].head(3))


df1:   screen_size   res_w   res_h  pixel_count
0   1920x1080  1920.0  1080.0    2073600.0
1   2880x1920  2880.0  1920.0    5529600.0
2     no_data  1920.0  1080.0    2073600.0
df2:   screen_size   res_w   res_h  pixel_count
0   2712x1220  2712.0  1220.0    3308640.0
1   3200x2400  3200.0  2400.0    7680000.0
2   2400x1080  2400.0  1080.0    2592000.0
df3:   screen_size   res_w   res_h  pixel_count
0   2504x1080  2504.0  1080.0    2704320.0
1    1280x800  1280.0   800.0    1024000.0
2   1920x1200  1920.0  1200.0    2304000.0


In [77]:
df3['discount_abs'] = df3['price_old'] - df3['price']
df3['discount_pct'] = safe_div(df3['discount_abs'], df3['price_old']) * 100

df3['cpu_power_proxy'] = df3['ghz_processor'] * df3['processor_cores']
df3['ram_per_core'] = safe_div(df3['ram'], df3['processor_cores'])

df3['storage_per_price'] = safe_div(df3['internal_storage'], df3['price'])
df3['ram_per_price'] = safe_div(df3['ram'], df3['price'])
df3['battery_per_gram'] = safe_div(df3['battery_capacity'], df3['weight_g'])

df3['total_camera_mp'] = df3['main_camera'] + df3['front_camera']

df3['high_refresh_rate'] = (df3['hz_screen'] >= 90).astype(int)
df3['screen_area_proxy'] = df3['screen_diagonal'] ** 2

df3['keyboard_flag'] = df3['keyboard'].astype(str).str.lower().str.contains(
    'да|yes|клавиатура|опционально|включена', regex=True
).astype(int)

df3['stylus_flag'] = df3['accessories_included'].astype(str).str.lower().str.contains(
    'стилус|stylus|pencil', regex=True
).astype(int)

df3['waterproof_flag'] = df3['waterproof'].astype(str).str.lower().str.contains(
    'да|yes|ip', regex=True
).astype(int)

new_cols_df3 = [
    'discount_abs', 'discount_pct', 'cpu_power_proxy', 'ram_per_core',
    'storage_per_price', 'ram_per_price', 'battery_per_gram',
    'total_camera_mp', 'high_refresh_rate', 'screen_area_proxy',
    'keyboard_flag', 'stylus_flag', 'waterproof_flag'
]

df3[new_cols_df3].head()


,discount_abs,discount_pct,cpu_power_proxy,ram_per_core,storage_per_price,ram_per_price,battery_per_gram,total_camera_mp,high_refresh_rate,screen_area_proxy,keyboard_flag,stylus_flag,waterproof_flag
0,148348.0,63.733223,12.8,0.75,0.003033,0.000071,16.733333,80.0,1,121.00,1,1,1
1,122133.0,60.660379,8.0,0.75,0.000808,0.000038,9.469697,13.3,0,102.01,0,0,0
2,241517.5,79.902569,16.0,1.00,0.002107,0.000132,16.210526,13.0,1,121.00,0,1,0
3,262182.0,81.403263,27.2,2.00,0.017096,0.000267,33.333333,48.0,1,102.01,1,0,1
4,697764.0,74.053901,26.4,1.00,0.000524,0.000033,17.413793,21.0,1,156.25,0,0,1


In [78]:
output_dir = os.path.join('data', 'cleaned')
os.makedirs(output_dir, exist_ok=True)

df1.to_csv(os.path.join(output_dir, 'laptops_cleaned.csv'), index=False)
df2.to_csv(os.path.join(output_dir, 'smartphones_cleaned.csv'), index=False)
df3.to_csv(os.path.join(output_dir, 'tablets_cleaned.csv'), index=False)

print(f'Saved to {output_dir}:')
print(f'  laptops_cleaned.csv     — {len(df1)} rows')
print(f'  smartphones_cleaned.csv — {len(df2)} rows')
print(f'  tablets_cleaned.csv     — {len(df3)} rows')

Saved to data\cleaned:
  laptops_cleaned.csv     — 5376 rows
  smartphones_cleaned.csv — 5382 rows
  tablets_cleaned.csv     — 5553 rows
